# Loading External Data

This tutorial demonstrates how to load external data into the `ParquetDataCatalog`, and then use this to run a one-shot backtest using a `BacktestNode`.

<div style="border:1px solid #ffcc00; padding:10px; margin-top:10px; margin-bottom:10px; background-color:#333333; color: #ffcc00;">
<b>Warning:</b> Intended to be run on bare metal (not in the JupyterLab Docker container).
</div>

In [1]:
import shutil
from decimal import Decimal
from pathlib import Path

import pandas as pd

from nautilus_trader.backtest.node import BacktestDataConfig
from nautilus_trader.backtest.node import BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.backtest.node import BacktestRunConfig
from nautilus_trader.backtest.node import BacktestVenueConfig
from nautilus_trader.config import ImportableStrategyConfig
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.model import BarType
from nautilus_trader.model import QuoteTick
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.persistence.wranglers import QuoteTickDataWrangler
from nautilus_trader.test_kit.providers import CSVTickDataLoader
from nautilus_trader.test_kit.providers import TestInstrumentProvider

In [5]:
DATA_DIR = "~/Downloads/Data/"

In [6]:
path = Path(DATA_DIR).expanduser() / "HISTDATA"
raw_files = [
    f for f in path.iterdir() if f.is_file() and (f.suffix == ".csv" or f.name.endswith(".csv.gz"))
]
assert raw_files, f"Unable to find any data files in directory {path}"
raw_files

[PosixPath('/Users/androidjk/Downloads/Data/HISTDATA/EURUSD_TICK_20200103.csv')]

In [7]:
# Load the first data file into a pandas DataFrame
df = CSVTickDataLoader.load(raw_files[0], index_col=0, datetime_format="%Y%m%d %H%M%S%f")
df.columns = ["bid_price", "ask_price"]

# Process quotes using a wrangler
EURUSD = TestInstrumentProvider.default_fx_ccy("EUR/USD")
wrangler = QuoteTickDataWrangler(EURUSD)

ticks = wrangler.process(df)

/Volumes/T9/projects/trade/nautilus_trader/nautilus_trader/persistence/loaders.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


In [14]:
df.head()

,bid_price,ask_price
20200103 000000000000,,
2020-01-03 00:01:00,1.1235,1.1237
2020-01-03 00:02:00,1.1233,1.1235
2020-01-03 00:03:00,1.1236,1.1238
2020-01-03 00:04:00,1.1238,1.1240
2020-01-03 00:05:00,1.1237,1.1239


In [8]:
CATALOG_PATH = Path.cwd() / "catalog"

# Clear if it already exists, then create fresh
if CATALOG_PATH.exists():
    shutil.rmtree(CATALOG_PATH)
CATALOG_PATH.mkdir()

catalog = ParquetDataCatalog(CATALOG_PATH)

In [12]:
EURUSD

CurrencyPair(id=EUR/USD.SIM, raw_symbol=EUR/USD, asset_class=FX, instrument_class=SPOT, quote_currency=USD, is_inverse=False, price_precision=5, price_increment=0.00001, size_precision=0, size_increment=1, multiplier=1, lot_size=1000, margin_init=0.03, margin_maint=0.03, maker_fee=0.00002, taker_fee=0.00002, info=None)

In [13]:
ticks[0]

QuoteTick(EUR/USD.SIM,1.12350,1.12370,1000000,1000000,1578009660000000000)

In [ ]:
catalog.write_data([EURUSD])
catalog.write_data(ticks)

In [10]:
# Verify instruments written to catalog
catalog.instruments()

[CurrencyPair(id=EUR/USD.SIM, raw_symbol=EUR/USD, asset_class=FX, instrument_class=SPOT, quote_currency=USD, is_inverse=False, price_precision=5, price_increment=0.00001, size_precision=0, size_increment=1, multiplier=1, lot_size=1000, margin_init=0.03, margin_maint=0.03, maker_fee=0.00002, taker_fee=0.00002, info=None)]

In [11]:
start = dt_to_unix_nanos(pd.Timestamp("2020-01-03", tz="UTC"))
end = dt_to_unix_nanos(pd.Timestamp("2020-01-04", tz="UTC"))

ticks = catalog.quote_ticks(instrument_ids=[EURUSD.id.value], start=start, end=end)
ticks[:10]

[QuoteTick(EUR/USD.SIM,1.12350,1.12370,1000000,1000000,1578009660000000000),
 QuoteTick(EUR/USD.SIM,1.12330,1.12350,1000000,1000000,1578009720000000000),
 QuoteTick(EUR/USD.SIM,1.12360,1.12380,1000000,1000000,1578009780000000000),
 QuoteTick(EUR/USD.SIM,1.12380,1.12400,1000000,1000000,1578009840000000000),
 QuoteTick(EUR/USD.SIM,1.12370,1.12390,1000000,1000000,1578009900000000000),
 QuoteTick(EUR/USD.SIM,1.12390,1.12410,1000000,1000000,1578009960000000000),
 QuoteTick(EUR/USD.SIM,1.12400,1.12420,1000000,1000000,1578010020000000000),
 QuoteTick(EUR/USD.SIM,1.12380,1.12400,1000000,1000000,1578010080000000000),
 QuoteTick(EUR/USD.SIM,1.12390,1.12410,1000000,1000000,1578010140000000000)]

In [15]:
instrument = catalog.instruments()[0]

venue_configs = [
    BacktestVenueConfig(
        name="SIM",
        oms_type="HEDGING",
        account_type="MARGIN",
        base_currency="USD",
        starting_balances=["1000000 USD"],
    ),
]

data_configs = [
    BacktestDataConfig(
        catalog_path=str(catalog.path),
        data_cls=QuoteTick,
        instrument_id=instrument.id,
        start_time=start,
        end_time=end,
    ),
]

strategies = [
    ImportableStrategyConfig(
        strategy_path="nautilus_trader.examples.strategies.ema_cross:EMACross",
        config_path="nautilus_trader.examples.strategies.ema_cross:EMACrossConfig",
        config={
            "instrument_id": instrument.id,
            "bar_type": BarType.from_str(f"{instrument.id.value}-15-MINUTE-BID-INTERNAL"),
            "fast_ema_period": 10,
            "slow_ema_period": 20,
            "trade_size": Decimal(1_000_000),
        },
    ),
]

config = BacktestRunConfig(
    engine=BacktestEngineConfig(strategies=strategies),
    data=data_configs,
    venues=venue_configs,
)

In [16]:
node = BacktestNode(configs=[config])

[result] = node.run()

2025-12-29T03:55:50.378095000Z [INFO] BACKTESTER-001.BacktestEngine: =================================================================
2025-12-29T03:55:50.380814000Z [INFO] BACKTESTER-001.BacktestEngine:  NAUTILUS TRADER - Automated Algorithmic Trading Platform
2025-12-29T03:55:50.380816000Z [INFO] BACKTESTER-001.BacktestEngine:  by Nautech Systems Pty Ltd.
2025-12-29T03:55:50.380817000Z [INFO] BACKTESTER-001.BacktestEngine:  Copyright (C) 2015-2025. All rights reserved.
2025-12-29T03:55:50.380817001Z [INFO] BACKTESTER-001.BacktestEngine: =================================================================
2025-12-29T03:55:50.380818000Z [INFO] BACKTESTER-001.BacktestEngine: 
2025-12-29T03:55:50.380818001Z [INFO] BACKTESTER-001.BacktestEngine: ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⣴⣶⡟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀
2025-12-29T03:55:50.380819000Z [INFO] BACKTESTER-001.BacktestEngine: ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣰⣾⣿⣿⣿⠀⢸⣿⣿⣿⣿⣶⣶⣤⣀⠀⠀⠀⠀⠀
2025-12-29T03:55:50.380819001Z [INFO] BACKTESTER-001.BacktestEngine: ⠀⠀⠀⠀⠀⠀⢀⣴⡇⢀⣾⣿⣿⣿⣿⣿⠀⣾⣿⣿⣿⣿⣿⣿⣿⠿⠓⠀⠀⠀⠀
2025-12-29T03:55

In [ ]:
result